# imports

In [ ]:
import sys
project_root = r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/UTvsXCT-preprocessing'
if project_root not in sys.path:
    sys.path.insert(0, project_root)  # insert, not append: site-packages holds an older non-editable copy
import importlib
from preprocess_tools import onlypores, datasetmaker, io, aligner, register
datasetmaker = importlib.reload(datasetmaker)
assert hasattr(datasetmaker, 'main_images_depth') and project_root in datasetmaker.__file__, datasetmaker.__file__
import numpy as np
import pandas as pd
import tifffile
from dbtools import dbtools as db
from dbtools import load as load
from pathlib import Path
import ast

# Path mapping

In [ ]:
LOCAL_DATA_ROOT = Path('/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente')
UNC_DATA_ROOT = r'\\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente'


def resolve_data_path(path, must_exist=True):
    path_str = str(path).strip()
    direct_path = Path(path_str)

    try:
        direct_exists = direct_path.exists()
    except OSError:
        direct_exists = False

    if direct_exists:
        return direct_path

    normalized_path = path_str.replace('\\', '/')
    normalized_unc_root = UNC_DATA_ROOT.replace('\\', '/')

    if normalized_path.lower().startswith(normalized_unc_root.lower()):
        relative_path = normalized_path[len(normalized_unc_root):].lstrip('/')
        local_path = LOCAL_DATA_ROOT / relative_path
        if local_path.exists() or not must_exist:
            return local_path

    if must_exist:
        raise FileNotFoundError(
            f"Could not resolve data path: {path_str}\n"
            f"Tried direct path and UNC mapping to: {LOCAL_DATA_ROOT}"
        )

    return direct_path

# Database connection

In [ ]:
try:
    conn = db.connect()
    print("Connected to the database")
except Exception as error:
    print(error)

# Select measurements to create the image datasets

## Measurement type

In [ ]:
measurementtypes_table = db.get_data_metadata('measurementtypes')

measurementtypes_table

In [ ]:
ut_type = 11

xct_type = 6

## Selecting UT measurements

In [ ]:
ut_measurements_table = db.get_data_metadata('measurements')

ut_measurements_table = ut_measurements_table[ut_measurements_table['measurementtype_id_measurement'] == ut_type]

ut_measurements_table

## Selecting XCT measurements

In [ ]:
xct_measurements_table = db.get_data_metadata('measurements')

xct_measurements_table = xct_measurements_table[xct_measurements_table['measurementtype_id_measurement'] == xct_type]

xct_measurements_table

## Getting registered pairs

In [ ]:
registrations_table = db.get_data_metadata('registrations')

registrations_table = registrations_table[registrations_table['reference_measurement_id_registration'].isin(ut_measurements_table['id_measurement'])]

registrations_table = registrations_table[registrations_table['registered_measurement_id_registration'].isin(xct_measurements_table['id_measurement'])]

registrations_table

## Sample and panel metadata

In [ ]:
sample_measurements_table = db.relation_metadata('samples', 'measurements', 'sample_measurements')
samples_table = db.get_data_metadata('samples')
panels_table = db.get_data_metadata('panels')

sample_measurements_table

In [ ]:
def lookup_sample_metadata(measurement_id):
    sample_rows = sample_measurements_table[sample_measurements_table['id_measurement'] == measurement_id]
    if sample_rows.empty:
        return {
            'sample_id': None,
            'sample_name': None,
            'panel_id': None,
            'panel_name': None,
        }

    sample_row = sample_rows.iloc[0]
    sample_id = sample_row.get('id_sample')
    sample_name = sample_row.get('name_sample')
    panel_id = sample_row.get('panel_id_sample')

    if pd.isna(panel_id) and sample_id is not None:
        matching_samples = samples_table[samples_table['id_sample'] == sample_id]
        if not matching_samples.empty:
            panel_id = matching_samples.iloc[0].get('panel_id_sample')

    panel_name = None
    if panel_id is not None and not pd.isna(panel_id):
        matching_panels = panels_table[panels_table['id_panel'] == panel_id]
        if not matching_panels.empty:
            panel_name = matching_panels.iloc[0].get('name_panel')

    return {
        'sample_id': sample_id,
        'sample_name': sample_name,
        'panel_id': panel_id,
        'panel_name': panel_name,
    }


registered_pairs = []

for _, registration in registrations_table.iterrows():
    reference_id = registration['reference_measurement_id_registration']
    registered_id = registration['registered_measurement_id_registration']

    reference_measurement = ut_measurements_table[ut_measurements_table['id_measurement'] == reference_id].iloc[0]
    registered_measurement = xct_measurements_table[xct_measurements_table['id_measurement'] == registered_id].iloc[0]
    sample_metadata = lookup_sample_metadata(reference_id)

    registered_pairs.append({
        'registration_id': registration['id_registration'],
        'reference_measurement_id': reference_id,
        'registered_measurement_id': registered_id,
        'reference_measurement_path': reference_measurement['file_path_measurement'],
        'registered_measurement_path': registered_measurement['file_path_measurement'],
        'sample_id': sample_metadata['sample_id'],
        'sample_name': sample_metadata['sample_name'],
        'panel_id': sample_metadata['panel_id'],
        'panel_name': sample_metadata['panel_name'],
    })

registered_pairs_df = pd.DataFrame(registered_pairs)

print(f"Found {len(registered_pairs_df)} registrations")
registered_pairs_df

# Dataset type selection

In [ ]:
datasettype_table = db.get_data('datasettypes')

datasettype_table

In [ ]:
datasettype = 12   # 'Depth Profile Dataset Type'
load_into_database = True   # False = files and manifests only


# Discard already computed datasets

In [ ]:
# Files are skipped only when the registration is in the database as its own dataset of this type
# (file_path = its image_manifest.csv). The global manifest links every registration, so it must not count.
registered_manifest_paths = set()
if datasettype is not None:
    try:
        datasets_table = db.get_data('datasets')
        registered_manifest_paths = set(datasets_table[datasets_table['datasettype_id_dataset'] == datasettype]['file_path_dataset'].astype(str))
    except Exception as e:
        print("Could not read the datasets table:", e)
print(f"{len(registered_manifest_paths)} datasets of type {datasettype} already in the database")


# Saving folder

In [ ]:
OUTPUT_ROOT = Path(r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/5MHz_DepthProfile')
only_samples = None  # e.g. ['Na_01_1'] for a test run; None = every registered pair

folder = OUTPUT_ROOT
folder.mkdir(parents=True, exist_ok=True)

# Image settings

In [ ]:
material_threshold = 0.8
stop_after_first = False
# Depth-profile labels
PATCH_UT = 5     # centred window side in UT pixels (odd), same linkage as the vvf_full label
NB_U = 100       # bins of normalised depth u (per XCT column, from its own front/back); group them later


# Resolutions

In [ ]:
xct_resolution = float(measurementtypes_table[measurementtypes_table['id_measurementtype'] == xct_type]['voxel_size_measurementtype'].values[0].split(' ')[0])
ut_resolution = float(measurementtypes_table[measurementtypes_table['id_measurementtype'] == ut_type]['x_resolution_measurementtype'].values[0].split(' ')[0])
print(f"XCT resolution: {xct_resolution}, UT resolution: {ut_resolution}")

# Dataset generation

In [ ]:
DEPTH_TARGETS = ['pore_u', 'mat_u', 'pore_d', 'mat_d', 'pore_u_window', 'mat_u_window', 'pore_d_window', 'mat_d_window',
                 'column_front', 'column_back', 'column_thickness', 'column_count', 'vvf_full', 'thickness', 'material_voxels']


def register_in_database(registration_id, manifest_path, map_shape):
    """Insert one per-registration dataset (datasets + metadata + dataset_registrations). Returns the dataset id or -1."""
    dataset_id = load.load_dataset(
        conn,
        datasettype_id=datasettype,
        file_path=str(manifest_path),
        rows=int(np.prod(map_shape)),
        patch_size='no_ut_patches',
        targets=DEPTH_TARGETS,
        reconstruction_shape=tuple(int(n) for n in map_shape),
        registration_ids=[int(registration_id)],
        description=f'Depth-profile dataset (per-column depth histograms, {NB_U} bins of u, {PATCH_UT}x{PATCH_UT} UT window), created with preprocess tools v 0.1.23',
    )
    if dataset_id == -1:
        print(f"!! Could not register registration {registration_id} in the database ({manifest_path}); fix and rerun")
    else:
        registered_manifest_paths.add(str(manifest_path))
    return dataset_id


excluded_samples = []

manifest_rows = []

for _, pair in registered_pairs_df.iterrows():
    reference_measurement_path_raw = pair['reference_measurement_path']
    registered_measurement_path_raw = pair['registered_measurement_path']
    registration_id = pair['registration_id']
    sample_name = pair['sample_name']
    sample_folder_name = sample_name if pd.notna(sample_name) else f"registration_{registration_id}"

    if sample_name in excluded_samples:
        print(f"Sample {sample_name} is in the exclusion list, skipping...")
        continue

    if only_samples is not None and sample_name not in only_samples:
        continue

    dataset_folder = folder / f"{sample_folder_name}" / f"registration_{registration_id}"
    registration_manifest_path = dataset_folder / 'image_manifest.csv'
    if registration_manifest_path.exists():
        existing_rows = pd.read_csv(registration_manifest_path)
        manifest_rows.extend(existing_rows.to_dict('records'))
        if load_into_database and datasettype is not None and str(registration_manifest_path) not in registered_manifest_paths:
            # files are complete but the database row is missing: register without recomputing
            map_shape = ast.literal_eval(existing_rows.loc[existing_rows['image_kind'] == 'material_voxels', 'image_shape'].iloc[0])
            print(f"Files of registration {registration_id} exist but are not in the database, registering...")
            register_in_database(registration_id, registration_manifest_path, map_shape)
        else:
            print(f"Depth dataset for registration {registration_id} already exists, skipping...")
        continue

    reference_measurement_path = resolve_data_path(reference_measurement_path_raw)
    registered_measurement_path = resolve_data_path(registered_measurement_path_raw)

    print(f"Creating depth dataset for registration {registration_id} with reference measurement {reference_measurement_path} and registered measurement {registered_measurement_path}")

    ut_volume = io.load_tif(reference_measurement_path)
    xct_volume = io.load_tif(registered_measurement_path)

    xct_volume = np.transpose(xct_volume, (1, 2, 0))
    ut_volume = np.transpose(ut_volume, (1, 2, 0))

    registration_parameters = registrations_table[registrations_table['id_registration'] == registration_id]['registration_matrix_registration'].values[0]
    registration_parameters = np.array(ast.literal_eval(registration_parameters))

    xct_volume = register.apply_registration(ut_volume, xct_volume, registration_parameters, ut_resolution, xct_resolution, parallel=True)

    _, frontwall, backwall = aligner.crop_walls(xct_volume)

    xct_volume = np.transpose(xct_volume, (2, 0, 1))
    ut_volume = np.transpose(ut_volume, (2, 0, 1))

    onlypores_volume, material_mask, _ = onlypores.onlypores(xct_volume, frontwall, backwall, min_size_filtering=8)

    dataset_folder.mkdir(parents=True, exist_ok=True)

    outputs = datasetmaker.main_images_depth(
        onlypores_volume,
        material_mask,
        ut_volume,
        xct_resolution,
        ut_resolution,
        material_threshold=material_threshold,
        patch_ut=PATCH_UT,
        nb_u=NB_U,
    )
    ut_image = outputs['ut_image']
    depth_rois = outputs['depth_rois']
    windows_df = outputs['windows_df']
    material_frontwall_z = depth_rois[0]['material_frontwall_z']
    material_backwall_z = depth_rois[0]['material_backwall_z']

    def bins_first(a):
        return np.moveaxis(a, -1, 0).astype(np.uint32)

    image_outputs = {
        'ut': ut_image,
        'vvf_full': outputs['vvf_full_map'].astype(np.float32),
        'material_voxels': outputs['material_voxels_map'].astype(np.uint32),
        'thickness': outputs['thickness_map'].astype(np.float32),
        'columns': outputs['columns'].astype(np.uint32),
        'front_mean': outputs['front_mean'], 'front_std': outputs['front_std'],
        'back_mean': outputs['back_mean'],
        'thickness_slices_mean': outputs['thickness_mean'], 'thickness_slices_std': outputs['thickness_std'],
        'pore_u': bins_first(outputs['pore_u']), 'mat_u': bins_first(outputs['mat_u']),
        'pore_d': bins_first(outputs['pore_d']), 'mat_d': bins_first(outputs['mat_d']),
        'pore_u_window': bins_first(outputs['pore_u_window']), 'mat_u_window': bins_first(outputs['mat_u_window']),
        'pore_d_window': bins_first(outputs['pore_d_window']), 'mat_d_window': bins_first(outputs['mat_d_window']),
    }
    image_files = {'ut': 'ut_volume.tif', 'vvf_full': 'vvf_full_map.tif', 'material_voxels': 'material_voxels_map.tif',
                   'thickness': 'thickness_map.tif', 'columns': 'column_count_map.tif',
                   'front_mean': 'column_front_mean_map.tif', 'front_std': 'column_front_std_map.tif',
                   'back_mean': 'column_back_mean_map.tif',
                   'thickness_slices_mean': 'column_thickness_mean_map.tif',
                   'thickness_slices_std': 'column_thickness_std_map.tif'}

    current_rows = []
    common = {
        'sample_id': pair['sample_id'], 'sample_name': sample_name,
        'panel_id': pair['panel_id'], 'panel_name': pair['panel_name'],
        'registration_id': registration_id,
        'reference_measurement_id': pair['reference_measurement_id'],
        'registered_measurement_id': pair['registered_measurement_id'],
        'reference_measurement_path': str(reference_measurement_path),
        'registered_measurement_path': str(registered_measurement_path),
        'reference_measurement_path_raw': reference_measurement_path_raw,
        'registered_measurement_path_raw': registered_measurement_path_raw,
        'ut_image_shape': tuple(ut_image.shape),
        'crop_walls_frontwall': int(frontwall), 'crop_walls_backwall': int(backwall),
        'material_frontwall_z': material_frontwall_z, 'material_backwall_z': material_backwall_z,
        'material_threshold': material_threshold,
        'xct_resolution': xct_resolution, 'ut_resolution': ut_resolution,
        'patch_ut': PATCH_UT, 'nb_u': NB_U,
    }
    for kind, array in image_outputs.items():
        path = dataset_folder / image_files.get(kind, f'{kind}_hist.tif')
        tifffile.imwrite(path, array, bigtiff=True)
        current_rows.append({'image_kind': kind, 'image_path': str(path), 'image_shape': tuple(array.shape),
                             'image_dtype': str(array.dtype), 'file_format': 'tif', **common})

    windows_path = dataset_folder / 'windows.parquet'
    windows_df.to_parquet(windows_path, index=False)
    current_rows.append({'image_kind': 'windows', 'image_path': str(windows_path), 'image_shape': tuple(windows_df.shape),
                         'image_dtype': 'table', 'file_format': 'parquet', **common})

    pd.DataFrame(current_rows).to_csv(registration_manifest_path, index=False)  # written last: marks the registration done
    manifest_rows.extend(current_rows)

    if load_into_database and datasettype is not None:
        register_in_database(registration_id, registration_manifest_path, outputs['material_voxels_map'].shape)

    print(f"Depth dataset for registration {registration_id} saved to {dataset_folder}")

    if stop_after_first:
        break

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = folder / 'image_dataset_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)
print(f"Saved depth dataset manifest to {manifest_path}")

# Global manifest: parent dataset linked to every registration in it (file_path must be new in the database)
if load_into_database and datasettype is not None and not manifest_df.empty:
    _all_paths = db.get_data('datasets')['file_path_dataset'].astype(str).values
    if str(manifest_path) in _all_paths:
        print(f"Global manifest {manifest_path} is already registered in the database, skipping")
    else:
        _registration_ids = sorted(int(r) for r in manifest_df['registration_id'].unique())
        load.load_dataset(
            conn,
            datasettype_id=datasettype,
            file_path=str(manifest_path),
            rows=int(len(manifest_df)),
            patch_size='no_ut_patches',
            targets=DEPTH_TARGETS,
            reconstruction_shape=(len(_registration_ids),),  # one entry per registration
            registration_ids=_registration_ids,
            description='Global depth-profile dataset manifest (all per-registration image manifests), created with preprocess tools v 0.1.23',
        )
manifest_df


# Recompose global manifest from per-registration manifests

Run this cell independently to rebuild `image_dataset_manifest.csv` from scratch by scanning all `image_manifest.csv` files already on disk. Useful if the global manifest is missing, corrupted, or out of sync with the per-registration files (e.g. after a partial run or manual deletion).

Set `recompose_folder` to the root images folder if running outside the notebook flow.

In [ ]:
# recompose_folder = folder  # change if running standalone

# per_registration_manifests = sorted(recompose_folder.glob('*/registration_*/image_manifest.csv'))
# print(f"Found {len(per_registration_manifests)} per-registration manifests")

# recomposed_df = pd.concat(
#     [pd.read_csv(p) for p in per_registration_manifests],
#     ignore_index=True,
# )

# recompose_manifest_path = recompose_folder / 'image_dataset_manifest.csv'
# recomposed_df.to_csv(recompose_manifest_path, index=False)

# print(f"Recomposed manifest saved to {recompose_manifest_path} ({len(recomposed_df)} rows)")
# recomposed_df